# Capstone Project 01 — Configurable Cash-Application Rule Engine

This notebook reconciles Bank Statement, ERP Open AR and Remittance Advice data using client-configurable rule priorities. It produces the required end states: **Open, Partial Match, Closed, UAC, UIC, and Query**.

The supplied Azure Foundry connection values should stay in a local `.env` file. This deterministic reconciliation engine does not send financial records to an LLM; an agent may be added later for extracting fields from unstructured remittances.

In [1]:
# Cell 1 — imports, safe environment configuration, and shared normalisation helpers.
from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal
from typing import Any
import os
import pandas as pd

pd.set_option('display.max_columns', None)

# Optional Foundry settings: provide these via environment/.env, never hard-code secrets.
FOUNDRY_PROJECT_ENDPOINT = os.getenv('FOUNDRY_PROJECT_ENDPOINT')
MODEL_DEPLOYMENT_NAME = os.getenv('MODEL_DEPLOYMENT_NAME', 'gpt-5')
MCP_SERVER_NAME = os.getenv('MCP_SERVER_NAME', 'Microsoft Learn MCP server')

def norm(value: Any) -> str:
    """Normalise text/date keys without changing the source data."""
    if value is None or pd.isna(value):
        return ''
    return str(value).strip().casefold()

def money(value: Any) -> Decimal:
    return Decimal(str(value or 0)).quantize(Decimal('0.01'))


In [2]:
# Cell 2 — client-specific rule configuration. Priority order controls which rule wins.
@dataclass(frozen=True)
class Rule:
    priority: int
    name: str
    source: str                 # BANK for 2-way, REMITTANCE for 3-way
    match_fields: tuple[str, ...]
    partial_action: str = 'Post remainder to UAC'

# Client configuration: replace/add this block per deal, without editing engine code.
CLIENT_RULES: dict[str, list[Rule]] = {
    'default_client': [
        Rule(1, 'PO + amount (2-way)', 'BANK', ('customer_name', 'po_number')),
        Rule(2, 'Delivery + amount (2-way)', 'BANK', ('customer_name', 'delivery_number')),
        Rule(3, 'Invoice + date + amount (2-way)', 'BANK', ('customer_name', 'invoice_number', 'invoice_date')),
        Rule(4, 'Invoice + amount (2-way)', 'BANK', ('customer_name', 'invoice_number')),
        Rule(5, 'Remittance PO + invoice + date (3-way)', 'REMITTANCE', ('customer_name', 'po_number', 'invoice_number', 'invoice_date')),
        Rule(6, 'Remittance PO + invoice (3-way)', 'REMITTANCE', ('customer_name', 'po_number', 'invoice_number')),
    ]
}

RULES = sorted(CLIENT_RULES['default_client'], key=lambda rule: rule.priority)
pd.DataFrame([{**rule.__dict__, 'match_fields': ' + '.join(rule.match_fields)} for rule in RULES])


,priority,name,source,match_fields,partial_action
0,1,PO + amount (2-way),BANK,customer_name + po_number,Post remainder to UAC
1,2,Delivery + amount (2-way),BANK,customer_name + delivery_number,Post remainder to UAC
2,3,Invoice + date + amount (2-way),BANK,customer_name + invoice_number + invoice_date,Post remainder to UAC
3,4,Invoice + amount (2-way),BANK,customer_name + invoice_number,Post remainder to UAC
4,5,Remittance PO + invoice + date (3-way),REMITTANCE,customer_name + po_number + invoice_number + i...,Post remainder to UAC
5,6,Remittance PO + invoice (3-way),REMITTANCE,customer_name + po_number + invoice_number,Post remainder to UAC


In [3]:
# Cell 3 — demo source records. In production, these three tables are read from the client's source files.
bank = pd.DataFrame([
    ('B001', 'Acme',    'PO-100', None,    None,      None,         1000, False),
    ('B002', 'Beta',    None,     'DN-200',None,      None,          750, False),
    ('B003', 'Gamma',   None,     None,    'INV-300', '2026-06-01', 1250, False),
    ('B004', 'Delta',   None,     None,    'INV-400', None,          900, False),
    ('B005', 'Epsilon', None,     None,    None,      None,          600, False),
    ('B006', 'Foxtrot', None,     None,    None,      None,          300, False),
    ('B007', 'Zeta',    None,     None,    None,      None,          150, False),
    ('B008', None,      None,     None,    None,      None,            0, False),
    ('B009', 'Eta',     None,     None,    'INV-900', None,          500, True),
    ('B010', 'Polaris', None,     None,    'INV-999', None,          250, False),
], columns=['bank_id', 'customer_name', 'po_number', 'delivery_number', 'invoice_number', 'invoice_date', 'amount', 'user_query'])

open_ar = pd.DataFrame([
    ('Acme',    'PO-100', None,     'INV-100', '2026-05-01', 1000),
    ('Beta',    None,     'DN-200', 'INV-200', '2026-05-15', 1000),
    ('Gamma',   None,     None,     'INV-300', '2026-06-01', 1250),
    ('Delta',   None,     None,     'INV-400', '2026-06-10', 1000),
    ('Epsilon', 'PO-500', None,     'INV-500', '2026-06-12',  600),
    ('Foxtrot', 'PO-600', None,     'INV-600', '2026-06-18',  300),
], columns=['customer_name', 'po_number', 'delivery_number', 'invoice_number', 'invoice_date', 'amount'])

remittance = pd.DataFrame([
    ('B005', 'Epsilon', 'PO-500', 'INV-500', '2026-06-12', 600),
    ('B006', 'Foxtrot', 'PO-600', 'INV-600', None,         300),
], columns=['bank_id', 'customer_name', 'po_number', 'invoice_number', 'invoice_date', 'amount'])


In [ ]:
# Cell 4 — reconciliation engine: selects rules in priority order and assigns an end state.

#Brief: Reconciles bank statements with open accounts receivable and remittance advice.

def candidate_ar(source: pd.Series, fields: tuple[str, ...], ar: pd.DataFrame) -> pd.DataFrame:
    candidates = ar.copy()
    for field in fields:
        candidates = candidates[candidates[field].map(norm) == norm(source.get(field))]
    return candidates

def reconcile(bank: pd.DataFrame, ar: pd.DataFrame, remittance: pd.DataFrame, rules: list[Rule]) -> pd.DataFrame:
    results: list[dict[str, Any]] = []
    remittance_by_bank = {row.bank_id: row for row in remittance.itertuples(index=False)}

    for record in bank.itertuples(index=False):
        row = pd.Series(record._asdict())
        base = {'bank_id': row.bank_id, 'bank_amount': money(row.amount), 'invoice_number': row.invoice_number}

        # Explicit user follow-up always wins: nothing is auto-applied.
        if row.user_query:
            results.append({**base, 'status': 'Query', 'rule': None, 'matched_invoice': None, 'matched_amount': Decimal('0.00'), 'uac_amount': Decimal('0.00'), 'reason': 'User requested more information'})
            continue
        # A blank bank statement cannot be identified.
        details = ['customer_name', 'po_number', 'delivery_number', 'invoice_number', 'invoice_date']
        if not any(norm(row[field]) for field in details) and money(row.amount) == 0:
            results.append({**base, 'status': 'UIC', 'rule': None, 'matched_invoice': None, 'matched_amount': Decimal('0.00'), 'uac_amount': Decimal('0.00'), 'reason': 'No information in bank statement'})
            continue

        matched = False
        for rule in rules:
            source = row if rule.source == 'BANK' else remittance_by_bank.get(row.bank_id)
            if source is None or any(not norm(source._asdict()[f] if hasattr(source, '_asdict') else source.get(f)) for f in rule.match_fields):
                continue
            source_series = pd.Series(source._asdict()) if hasattr(source, '_asdict') else source
            candidates = candidate_ar(source_series, rule.match_fields, ar)
            if candidates.empty:
                continue
            ar_row = candidates.iloc[0]
            bank_amount, ar_amount = money(row.amount), money(ar_row.amount)
            applied = min(bank_amount, ar_amount)
            difference = abs(bank_amount - ar_amount)
            status = 'Closed' if difference == 0 else 'Partial Match'
            results.append({**base, 'status': status, 'rule': f'P{rule.priority}: {rule.name}', 'matched_invoice': ar_row.invoice_number, 'matched_amount': applied, 'uac_amount': difference, 'reason': 'Exact amount match' if status == 'Closed' else rule.partial_action})
            matched = True
            break

        if not matched:
            has_reference = any(norm(row[f]) for f in ['po_number', 'delivery_number', 'invoice_number'])
            status = 'Open' if has_reference else 'UAC'
            reason = 'No Open AR candidate matched bank statement' if status == 'Open' else 'No invoice details in bank statement and no remittance advice'
            results.append({**base, 'status': status, 'rule': None, 'matched_invoice': None, 'matched_amount': Decimal('0.00'), 'uac_amount': money(row.amount) if status == 'UAC' else Decimal('0.00'), 'reason': reason})

    return pd.DataFrame(results)

results = reconcile(bank, open_ar, remittance, RULES)
results


,bank_id,bank_amount,invoice_number,status,rule,matched_invoice,matched_amount,uac_amount,reason
0,B001,1000.00,NaN,Closed,P1: PO + amount (2-way),INV-100,1000.00,0.00,Exact amount match
1,B002,750.00,NaN,Partial Match,P2: Delivery + amount (2-way),INV-200,750.00,250.00,Post remainder to UAC
2,B003,1250.00,INV-300,Closed,P3: Invoice + date + amount (2-way),INV-300,1250.00,0.00,Exact amount match
3,B004,900.00,INV-400,Partial Match,P4: Invoice + amount (2-way),INV-400,900.00,100.00,Post remainder to UAC
4,B005,600.00,NaN,Closed,P5: Remittance PO + invoice + date (3-way),INV-500,600.00,0.00,Exact amount match
5,B006,300.00,NaN,Closed,P6: Remittance PO + invoice (3-way),INV-600,300.00,0.00,Exact amount match
6,B007,150.00,NaN,UAC,NaN,NaN,0.00,150.00,No invoice details in bank statement and no re...
7,B008,0.00,NaN,UIC,NaN,NaN,0.00,0.00,No information in bank statement
8,B009,500.00,INV-900,Query,NaN,NaN,0.00,0.00,User requested more information
9,B010,250.00,INV-999,Open,NaN,NaN,0.00,0.00,No Open AR candidate matched bank statement


In [ ]:
# Cell 5 — acceptance validation and a business-level reconciliation summary.
# Acceptance check: every requested end state is represented.
#Brief: Validates that the demo data covers all six acceptance states.

expected_statuses = {'Open', 'Partial Match', 'Closed', 'UAC', 'UIC', 'Query'}
assert expected_statuses <= set(results.status), 'Demo data must cover every acceptance state'

summary = (results.groupby('status', sort=False)
           .agg(records=('bank_id', 'size'), bank_value=('bank_amount', 'sum'), uac_value=('uac_amount', 'sum'))
           .reindex(['Open', 'Partial Match', 'Closed', 'UAC', 'UIC', 'Query'])
           .fillna(0))
print('Acceptance check passed: all six end states were assigned.')
summary


Acceptance check passed: all six end states were assigned.


,records,bank_value,uac_value
status,,,
Open,1,250.00,0.00
Partial Match,2,1650.00,350.00
Closed,4,3150.00,0.00
UAC,1,150.00,150.00
UIC,1,0.00,0.00
Query,1,500.00,0.00


## Result interpretation

- **Closed:** B001 (priority 1), B003 (priority 3), B005 (priority 5) and B006 (priority 6) fully reconcile.
- **Partial Match:** B002 and B004 apply the available amount; their remaining $250 and $100 are posted to UAC.
- **Open:** B010 has usable invoice data but no matching Open AR record.
- **UAC / UIC / Query:** B007, B008 and B009 exercise the required exception states.

To onboard another client, define its ordered `Rule` list in `CLIENT_RULES` and invoke `reconcile` with that client's rules.